# Preprocessing

In [ ]:
import json
from pprint import pprint

In [ ]:
# read csv file
import os
import pandas as pd


df = pd.read_csv('o4mini_annotation_final.csv')
df.head()


In [ ]:
df.info(verbose=True)

In [ ]:
# print out all the columns for one row from the dataframe
row = df.iloc[1089]
for col in df.columns:
    print(f"{col}: {row[col]}") 

In [ ]:
# get columns from df as new df
data = df[['vaers_id','symptom_text', 'symptom_list','llm_result','llm_result_formatted','final_corrected_version','final_result']]
data.info()

In [ ]:
# check if "no temporal information" in final_corrected_version or llm_result_formatted if final_corrected_version is NaN and store in new column "no_temporal_info"
data['no_temporal_info'] = data.apply(lambda row:
    'no temporal information' in str(row['final_corrected_version']).lower() or
    ('final_corrected_version' in str(row['final_corrected_version']).lower() if
    pd.isna(row['final_corrected_version']) else False) or
    'no temporal information' in str(row['llm_result_formatted']).lower(),
    axis=1
)
data['no_temporal_info'].value_counts()
# get rows where no_temporal_info is False as new df
data_with_temporal_info = data[data['no_temporal_info'] == False]
data_with_temporal_info.info()
# we have to remove final_result value 2 from data_with_temporal_info
data_with_temporal_info = data_with_temporal_info[data_with_temporal_info['final_result'] != 2]
data_with_temporal_info.info()

In [ ]:
# find data with no temporal info
data_no_temporal_info = data[data['no_temporal_info'] == True]
data_no_temporal_info = data_no_temporal_info[data_no_temporal_info['final_result'] != 2]

data_no_temporal_info.info()

In [ ]:
# take 20 each from data_with_temporal_info and data_no_temporal_info
data_sampled_with_temporal_info = data_with_temporal_info.sample(n=200, random_state=67)
data_sampled_no_temporal_info = data_no_temporal_info.sample(n=100, random_state=67)
# concatenate the two dataframes
data_sampled = pd.concat([data_sampled_with_temporal_info, data_sampled_no_temporal_info])
data_sampled.info()
# save to csv
# data_sampled.to_csv('sample_40.csv', index=False)

In [ ]:
# make some sample that we can use to test
# data_sampled_with_temporal_info.to_csv('sample_our_method_with_temporal_info_200.csv', index=False)
# data_sampled_no_temporal_info.to_csv('sample_our_method_no_temporal_info_100.csv', index=False)
# data_sampled.to_csv('sample_our_method_total_300.csv', index=False)

In [ ]:
# now lets take 40 random samples from data_with_temporal_info with random_state=2
sample = data_with_temporal_info
sample.info()

# Prompt body

In [ ]:
from string import Template
import re
temporal_id_prompt_tpl = Template(r"""
You are an expert medical analyst. Your task is to determine if there is a temporal progression of different adverse events in the clinical notes. 

Definition: Temporal progression means a sequence where a symptom not previously mentioned appears after a previously mentioned symptom. The new symptoms must be distinct from any symptoms already listed in the clinical notes. 

What does not constitute temporal progression: 
- Concurrent Onset: All symptoms begin at the same time. 
- Symptom Resolution: A symptom disappears or improves. 
- Fluctuating Severity: A symptom gets better and worse. 
- Persistence: A symptom persists. 
- Inferred Timelines: Timelines based on symptom durations alone. For example, "Headache for 4 days, fever for 3 days" does not demonstrate temporal progression. 
- Assumptions: Any assumptions about the order of events if not directly stated in the text. 
- Re-emergence of Previous Symptoms: Re-emergence or continuation of previously reported symptoms, even with changes in severity, does not constitute temporal progression. The sequence must involve entirely new symptoms. 

Focus: Focus on whether different adverse events appear at different times, as explicitly stated in the clinical notes. Do not consider changes to the same adverse event over time. Pay close attention to whether the clinical notes introduce a symptom that has not been mentioned previously. 

Example of what does not constitute temporal progression: "Patient reported fatigue for 5 days and a cough for 2 days, then the fatigue returned." The return of fatigue is not a new event. Another example: "Patient reported headache and nausea, then the headache persisted the next day. This is not temporal progression." 

Clinical Notes:
$note

Output:
- If a temporal progression of different adverse events is identified, return exactly: Temporal information found
- If no temporal progression of different adverse events is found, return exactly: No temporal information found
""").substitute
# Keep your existing heuristic
_TEMPORAL_MARKERS_RE = re.compile(
    r"\b(then|after|later|subsequent(?:ly)?|the\s+next\s+day|the\s+following\s+day|day\s*\d+|week\s*\d+|post[-\s]|prior\s+to|initially|thereafter|by\s+day|on\s+day|on\s+\d{1,2}/\d{1,2}/\d{2,4})\b",
    flags=re.IGNORECASE
)

In [ ]:
# first version of prompt template

from string import Template

generation_prompt =  Template(""" 
Ignore previous conversations. 
 
TASK: 
Temporally order the provided list of adverse events based on their sequence of appearance in the clinical notes and extract the most specific mention for each adverse event from the notes. 
 
Clinical Notes: $symptom_text

 
Adverse Events to Temporally Order: $symptom_list
 
 
PROCESSING INSTRUCTIONS: 
- Start with the provided adverse events list and reorder it based on the sequence implied in the clinical notes. 
- If several adverse effects are mentioned together without a clear temporal order, group them into the same block (inside a single dictionary). 
- For each adverse event, extract the closest matching phrase from the clinical notes; if an adverse event is not mentioned, assign "none" as its value. 
 
IMPORTANT RULES: 
- No Invention: Never add, remove, or modify symptom names from the provided list. 
- Mention Each Symptom Once: Mention each symptom only once — at the first time it appears or becomes relevant. 
- Specific vs Generic Terms: When a phrase matches both specific and generic symptoms, assign it to the specific term only. If "Lip swelling" or "Pharyngeal swelling" is matched, do NOT include the generic "Swelling" unless there is a separate, explicit mention of general swelling elsewhere. 
- Temporal Evidence Required: If multiple symptoms are mentioned across multiple sentences without clear timeline separation, group them together. Do NOT treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", specific dates or times. 
- Unmentioned Symptoms: Group all symptoms with no original mention together in a separate block with "none" as their value. This block should be placed after all the temporally ordered groups. 
 
OUTPUT FORMAT: 
Return your answer only in valid JSON format — using a list of dictionaries to represent temporal progression and grouping, as shown below: 
{[
  {"Erythema": ["redness in neck"]},
  {
    "Pain in extremity": ["sore arm"],
    "Pruritus": ["itchy feeling"]
  },
  {"Swelling": ["mild arm swelling"]}
]}
                                 
# When revising, consider both the feedback (if any) and the previous attempt result (if provided).
# Keep correct parts from prior attempts, but fix issues based on feedback.

$prev_result_block
$feedback_block
""").substitute

verify_prompt_anchor_tpl = Template("""
Ignore previous conversations.

You are a temporal consistency verifier for a candidate adverse-event timeline.

Clinical Notes:
$symptom_text

Adverse Events List (canonical keys):
$symptom_list

Candidate Timeline JSON (a list of timepoints; each timepoint is a dict from canonical key -> "none" or extracted text/list):
$initial_result

Goal:
Evaluate ONLY the temporal sequence quality (ordering + grouping). Ignore whether extracted mention strings are exact quotes.

ANCHOR-ONLY verification method (do NOT output anchors or reasoning):
1) Infer a dominant time anchor Ai for each timepoint i using explicit dates/day counts and strong temporal phrases in the note.
   - Ai can be an explicit date/day (e.g., 3/6, March 8, Day 2, post-vax day 1), a clear stage boundary (e.g., ED visit/admitted/discharged/returned/follow-up), or Unknown.
2) Chronology:
   - Penalize only CLEAR contradictions where the order of timepoints conflicts with explicit anchor order in the note.
3) Grouping validity (using anchors):
   - Separation (over-merge): If a single timepoint clearly mixes events tied to different anchors, it is invalid.
   - Cohesion (over-split): If two adjacent timepoints clearly share the same anchor and there is no clear stage/transition boundary, the split is likely unnecessary.
4) Minimality/support:
   - Prefer the simplest timeline consistent with anchors (avoid extra timepoints without distinct anchor justification).

Scoring (universal 0–5, strict and conservative):
Start at 5. Subtract 1 for each CLEAR violation (max 5):
- Clear chronology contradiction with explicit anchors
- Clear over-merge: mixed anchors within one timepoint
- Clear over-split: redundant adjacent timepoints without distinct anchor/stage justification
- Clear lack of temporal support: too many timepoints despite only one anchor/stage in the note
- Overall timeline clearly inconsistent with temporal cues in the note

Conservatism rule:
If the note is ambiguous and you cannot identify a CLEAR violation, do NOT subtract points. Do NOT guess.

Feedback requirement (only when score < 4):
Return an actionable edit plan with AT MOST 2 operations, using ONLY the following operations:
- SWAP(g_i,g_j)
- MERGE(g_i,g_{i+1})
- MOVE(<symptom_key>, g_from, g_to)
- SPLIT(g_i, move_keys=[<symptom_key>, ...], to=NEW_AFTER_i)

Guidelines for feedback:
- Use 0-based group indices (g0, g1, ...).
- Refer to events ONLY by canonical symptom keys (not free-text).
- Prefer minimal edits (1–2 ops) that fix the most severe issue first.
- If you cannot propose a confident edit, set score to 3 and explain briefly what is ambiguous.

Output ONLY valid JSON:
- If score >= 3: {"score": <int>, "feedback": ""}
- If score < 3:  {"score": <int>, "feedback": "ASPECT=<CHRONOLOGY|GROUPING|MINIMALITY>; OP=<...>; OP=<...>"}
Do not output anchors, quotes, or any reasoning.

""").substitute


In [ ]:
# --------- Prompt templates ----------

verify_prompt_tpl = Template("""
Given the original text and extracted symptoms below:

Original Text:
$symptom_text

Symptoms to Extract:
$symptom_list

Current Result (JSON):
$initial_result

Scoring rubric (0–5), +1 each if:
1) The JSON is valid and groups are earliest→latest,
2) **Non-mentioned symptoms** should be presented as "none" in the **last** group,
3) Every symptom in the list appears exactly once overall,
4) Symptoms grouped together occur around the same time,
5) Group ordering follows the text's temporal cues.

Return ONLY one of the following JSON objects:
- If score >= 3: {"score": <int>, "feedback": ""}   # feedback can be empty string
- If score < 3:  {"score": <int>, "feedback": "<specific, actionable fixes>"}
""").substitute

# Modeling

In [ ]:
import time
import random
import os, json, re, ast
from typing import Any, Optional, Dict, List, Tuple
import pandas as pd
from anthropic import Anthropic
import anthropic
# ====== concurrency_runner.py ======
import time, random, threading, traceback
import concurrent.futures, multiprocessing as mp
from typing import Any, Callable, Dict, List, Optional, Tuple

# ---------- 1) Global RPM limiter (shared by ALL threads/calls) ----------
class RateLimiter:
    """Simple token-bucket limiter in RPM units."""
    def __init__(self, rpm: float, burst: int = 1):
        self.capacity = max(1, burst)
        self.tokens = float(self.capacity)
        self.refill_per_sec = float(rpm) / 60.0
        self.lock = threading.Lock()
        self.last = time.monotonic()

    def acquire(self):
        while True:
            with self.lock:
                now = time.monotonic()
                elapsed = now - self.last
                self.last = now
                # refill tokens
                self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_per_sec)
                if self.tokens >= 1.0:
                    self.tokens -= 1.0
                    return
            time.sleep(0.01)

# Put this somewhere central and import it in your call_llm file:
GLOBAL_RPM_LIMITER: Optional[RateLimiter] = None  # set later with your org RPM


client = Anthropic(api_key="") 
# prompt being used 
gen_prompt = generation_prompt
ver_prompt = verify_prompt_anchor_tpl

# --- REPLACE: format_check_and_fix with allowed key filtering -----------------
def _parse_json_lenient(x: Any) -> Optional[Any]:
    
    if isinstance(x, (dict, list)):
        return x
    if not isinstance(x, str):
        return None
    s = x.strip().strip('"')
    if s == "No temporal information found":
        return s
    if s.startswith("```json") and s.endswith("```"):
        s = s[len("```json"):-3].strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", s, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    start = min([p for p in (s.find("{"), s.find("[")) if p != -1], default=-1)
    if start != -1:
        for end in range(len(s) - 1, start, -1):
            if s[end] in "}]":
                chunk = s[start:end+1]
                try:
                    return json.loads(chunk)
                except Exception:
                    continue
    return None


def format_check_and_fix(
    raw_or_obj: Any,
    symptom_list: List[str],
    *,
    accept_envelope_keys: Tuple[str, ...] = ("temporal_result", "result", "output"),
    generic_terms: Tuple[str, ...] = ("Condition aggravated",),
    none_tokens: Tuple[str, ...] = ("none",),
) -> Dict[str, Any]:
    """
    Canonicalizes the model output to List[Dict[str, 'none' | List[str]]], earliest->latest,
    with all 'none' consolidated in the last group and full coverage for the (normalized) symptom_list.
    Drops unknown keys not in symptom_list.
    """
    # normalize and gate allowed keys
    symptom_list = normalize_symptom_list(symptom_list)
    allowed = set(symptom_list)

    report_issues: List[str] = []
    changes = {
        "unwrapped_envelope": False,
        "coerced_non_list": 0,
        "removed_generic_terms": 0,
        "dropped_unknown_keys": 0,   # <--- NEW: track drops
        "moved_none_to_last": 0,
        "deduped_duplicates": 0,
        "added_missing_none": 0,
        "removed_empty_groups": 0,
        "normalized_values": 0,
    }

    obj = _parse_json_lenient(raw_or_obj)
    if obj is None:
        report_issues.append("parse_failed")
        fixed = [{sym: "none" for sym in symptom_list}]
        changes["added_missing_none"] = len(symptom_list)
        return {"fixed": fixed, "report": {"issues": report_issues, "changes": changes,
                                           "counts": {"groups": 1, "symptoms": len(symptom_list)}}}
    if obj == "No temporal information found":
        fixed = [{sym: "none" for sym in symptom_list}]
        return {"fixed": fixed, "report": {"issues": [], "changes": {"added_missing_none": len(symptom_list)},
                                           "counts": {"groups": 1, "symptoms": len(symptom_list)}}}

    payload = obj
    if isinstance(obj, dict):
        for k in accept_envelope_keys:
            if k in obj:
                payload = obj[k]
                changes["unwrapped_envelope"] = True
                break
    if isinstance(payload, dict):
        payload = [payload]; changes["coerced_non_list"] += 1
    if not isinstance(payload, list):
        report_issues.append("not_list_after_unwrap"); payload = [{}]

    generic_set = set(generic_terms)
    none_set = set(t.lower() for t in none_tokens)

    norm_groups: List[Dict[str, Any]] = []
    for grp in payload:
        if not isinstance(grp, dict):
            changes["coerced_non_list"] += 1
            continue
        new_grp: Dict[str, Any] = {}
        for k, v in grp.items():
            # DROP unknown keys immediately
            if k not in allowed:
                if k in generic_set:
                    changes["removed_generic_terms"] += 1
                else:
                    changes["dropped_unknown_keys"] += 1
                continue

            # normalize value -> "none" OR non-empty List[str]
            val_is_none = False
            if v is None:
                val_is_none = True
            elif isinstance(v, str):
                val_is_none = v.strip().lower() in none_set
            elif isinstance(v, list):
                vv = [str(t).strip() for t in v if str(t).strip()]
                vv = [t for t in vv if t.lower() not in none_set]
                if len(vv) == 0:
                    val_is_none = True
                else:
                    v = vv
            else:
                v = [str(v).strip()] if str(v).strip() else "none"
                val_is_none = (v == "none")

            if val_is_none:
                new_grp[k] = "none"; changes["normalized_values"] += 1
            else:
                if isinstance(v, str):
                    v = [v.strip()] if v.strip() else []
                if isinstance(v, list):
                    v = [t for t in v if t]
                if not v:
                    new_grp[k] = "none"; changes["normalized_values"] += 1
                else:
                    new_grp[k] = v

        if new_grp:
            norm_groups.append(new_grp)
        else:
            changes["removed_empty_groups"] += 1

    # Keep earliest non-'none' per symptom across groups
    first_non_none_idx: Dict[str, int] = {}
    for idx, grp in enumerate(norm_groups):
        for k, v in grp.items():
            if v != "none" and k not in first_non_none_idx:
                first_non_none_idx[k] = idx

    cleaned_groups: List[Dict[str, Any]] = []
    for i, grp in enumerate(norm_groups):
        out: Dict[str, Any] = {}
        for k, v in grp.items():
            if v == "none":
                changes["moved_none_to_last"] += 1
                continue
            if first_non_none_idx.get(k, i) == i:
                out[k] = v
            else:
                changes["deduped_duplicates"] += 1
        if out:
            cleaned_groups.append(out)

    present_syms = set().union(*(set(g.keys()) for g in cleaned_groups)) if cleaned_groups else set()
    all_syms = set(symptom_list)
    missing = sorted(list(all_syms - present_syms))
    if missing:
        cleaned_groups.append({k: "none" for k in missing})
        changes["added_missing_none"] += len(missing)

    if not cleaned_groups:
        cleaned_groups = [{sym: "none" for sym in symptom_list}]
        changes["added_missing_none"] = len(symptom_list)

    counts = {
        "groups": len(cleaned_groups),
        "symptoms": len(symptom_list),
        "present_non_none": len(present_syms),
        "missing_as_none": changes["added_missing_none"],
    }

    return {"fixed": cleaned_groups, "report": {"issues": report_issues, "changes": changes, "counts": counts}}

# ---------- Helper: permissive JSON parser ----------
def parse_json_best_effort(text: str) -> Optional[Any]:
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    start = min([p for p in (text.find("{"), text.find("[")) if p != -1], default=-1)
    if start != -1:
        for end in range(len(text) - 1, start, -1):
            if text[end] in "}]":
                chunk = text[start:end + 1]
                try:
                    return json.loads(chunk)
                except Exception:
                    continue
    return None


# ---------- LLM call (Responses API)  ----------
def _split_system_and_messages(messages: List[Dict[str, str]]) -> Tuple[Optional[str], List[Dict[str, str]]]:
    """Claude Messages API uses a separate `system` field (no system role inside `messages`)."""
    system_parts: List[str] = []
    out_messages: List[Dict[str, str]] = []
    for m in messages or []:
        role = (m.get("role") or "user").strip()
        content = m.get("content") or ""
        if role == "system":
            if content:
                system_parts.append(str(content))
        elif role in ("user", "assistant"):
            out_messages.append({"role": role, "content": str(content)})
        else:
            # Unknown role — treat as user content to avoid API errors.
            out_messages.append({"role": "user", "content": f"[{role}] {content}"})
    system = "\n\n".join(system_parts).strip() if system_parts else None
    if not out_messages:
        out_messages = [{"role": "user", "content": ""}]
    return system, out_messages


def _anthropic_text_from_response(resp: Any) -> str:
    """Concatenate text content blocks from Anthropic responses."""
    try:
        parts = []
        for b in getattr(resp, "content", []) or []:
            b_type = getattr(b, "type", None) if not isinstance(b, dict) else b.get("type")
            if b_type == "text":
                parts.append(getattr(b, "text", None) if not isinstance(b, dict) else b.get("text", ""))
        return "".join([p for p in parts if p is not None])
    except Exception:
        # fallback
        return str(getattr(resp, "content", resp))


def call_llm(
    messages: List[Dict[str, str]],
    model: str = None,
    text: Optional[Dict[str, Any]] = None,   # kept for compatibility; Claude uses prompting (or beta `output_format`)
    temperature: float = 0,
    max_output_tokens: int = 512,
    delay: float = 3.5,
    backoff: float = 2.0,
    retries: int = 5,
    print_debug: bool = False
) -> Tuple[str, Any]:
    """
    Minimal OpenAI->Claude adapter.
    Returns: (out_text, raw_response)
    """
    global GLOBAL_RPM_LIMITER
    GLOBAL_RPM_LIMITER = RateLimiter(50)   # default to 50 RPM if not set elsewhere
    if model is None:
        model = "claude-sonnet-4-5"


    for attempt in range(retries):
        try:
            system, msgs = _split_system_and_messages(messages)

            kwargs: Dict[str, Any] = dict(
                model=model,
                messages=msgs,
                max_tokens=max_output_tokens,
                temperature=temperature,
            )
            if system:
                kwargs["system"] = system

            # If you want schema-guaranteed JSON, Claude supports "structured outputs" in beta for
            # specific models. This notebook keeps prompting-only output for minimal changes.
            resp = client.messages.create(**kwargs)
            out_text = _anthropic_text_from_response(resp)

            if print_debug:
                print("\n================ LLM CALL =================")
                print(f"Model: {model}")
                print("---- RAW RESPONSE ----")
                print(resp)
                # Anthropic usage lives under resp.usage when present
                usage = getattr(resp, "usage", None)
                print("---- USAGE ----")
                try:
                    print(usage.model_dump() if usage else "(no usage)")
                except Exception:
                    print(usage)
                print("===========================================\n")

            return out_text, resp

        except Exception as e:
            # Anthropic SDK raises typed exceptions, but we keep this handler broad to be version-tolerant.
            status = getattr(e, "status_code", None) or getattr(e, "status", None)
            name = e.__class__.__name__
            if status == 429 or "RateLimit" in name:
                print(f"RateLimitError. Retrying in {delay} sec...")
            elif "Timeout" in name or status == 408:
                print(f"Timeout. Retrying in {delay} sec...")
            else:
                print(f"Other error: {str(e)}. Retrying in {delay} sec...")

            time.sleep(delay)
            delay *= (backoff + random.uniform(0, 1))  # jitter

    return "Error: Failed after retries", None

def normalize_symptom_list(symptom_list):
    """
    Accept Python list/tuple/set, JSON string list, or Python-list string (single quotes).
    Returns a clean, deduplicated List[str].
    """
    # If it's already a list-like, keep it
    if isinstance(symptom_list, (list, tuple, set)):
        seq = list(symptom_list)
    elif isinstance(symptom_list, str):
        s = symptom_list.strip()
        # 1) Try JSON first (["A","B"])
        try:
            maybe = json.loads(s)
            seq = maybe if isinstance(maybe, (list, tuple, set)) else [maybe]
        except Exception:
            # 2) Try Python literal (['A', 'B'])
            try:
                maybe = ast.literal_eval(s)
                seq = maybe if isinstance(maybe, (list, tuple, set)) else [maybe]
            except Exception:
                # 3) Last-resort: bracketed CSV-ish string
                if s.startswith("[") and s.endswith("]"):
                    inner = s[1:-1]
                    parts = [p.strip().strip('"').strip("'") for p in inner.split(",") if p.strip()]
                    seq = parts if parts else [s]
                else:
                    seq = [s]
    else:
        # unknown type → coerce to single string
        seq = [str(symptom_list)]

    # Flatten, coerce to str, trim, dedupe
    out, seen = [], set()
    for x in seq:
        if isinstance(x, (list, tuple, set)):
            for y in x:
                y = "" if y is None else str(y).strip()
                if y and y not in seen:
                    out.append(y); seen.add(y)
        else:
            y = "" if x is None else str(x).strip()
            if y and y not in seen:
                out.append(y); seen.add(y)
    return out


def temporal_regex_heuristic(text: str) -> Dict[str, Any]:
    if not text:
        return {"has_progression": False, "signals": [], "confidence": 0.0}
    hits = list({m.group(0) for m in _TEMPORAL_MARKERS_RE.finditer(text)})
    return {
        "has_progression": bool(hits),
        "signals": hits,
        "confidence": 0.6 if hits else 0.1
    }

def temporal_id_once(note_text: str,
                     model: str = "gpt-4.1",
                     min_confidence: float = 0.5,
                     print_debug: bool = True) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Uses your binary-output prompt. Maps to decision dict:
      {"has_progression": bool, "signals": [...], "confidence": float}
    """
    user_prompt = temporal_id_prompt_tpl(note=note_text)
    messages = [
        {"role": "system", "content": "Return exactly one of the two strings requested. No extra words."},
        {"role": "user", "content": user_prompt},
    ]

    # IMPORTANT: ask for plain text (not JSON)
    raw, resp = call_llm(
        messages,
        model=model,
        text={"format": {"type": "text"}},   # ensure we get a plain string back
        temperature=0,
        max_output_tokens=16,
        print_debug=print_debug
    )

    raw_norm = (raw or "").strip().lower()

    # Default signals via heuristic (useful for logging even if model returns the string)
    heuristic = temporal_regex_heuristic(note_text)

    if "no temporal information found" in raw_norm:
        decision = {
            "has_progression": False,
            "signals": heuristic["signals"],   # keep for audit
            "confidence": 1.0                  # strong because model matched exact string
        }
        used_fallback = False
    elif "temporal information found" in raw_norm:
        decision = {
            "has_progression": True,
            "signals": heuristic["signals"],
            "confidence": 1.0
        }
        used_fallback = False
    else:
        # Model didn’t follow the strict output → fallback to heuristic
        decision = heuristic
        # Conservative: if heuristic fired, keep confidence; else very low
        used_fallback = True

    # Apply your confidence gate
    if decision["confidence"] < min_confidence:
        decision["has_progression"] = False

    if print_debug:
        print("---- TEMPORAL ID (decision) ----")
        print(json.dumps(decision, ensure_ascii=False, indent=2))
        print(f"raw_norm='{raw_norm}'  used_fallback={used_fallback}")
        print("--------------------------------\n")

    return decision, {"raw": raw, "used_fallback": used_fallback, "api_resp": resp}

# ---------- Agent (ITERATIVE, prints every step) ----------
class TemporalExtractionAgent:
    def __init__(self, 
                 model_generate: str = os.environ.get("CLAUDE_MODEL", "claude-3-5-sonnet-20240620"), 
                 model_verify: Optional[str] = None, 
                 max_iterations: int = 3,
                 use_temporal_gate: bool = True,
                 temporal_conf_threshold: float = 0.5):
        self.model_generate = model_generate
        self.model_verify = model_verify or model_generate
        self.max_iterations = max_iterations
        self.use_temporal_gate = use_temporal_gate
        self.temporal_conf_threshold = temporal_conf_threshold

    def generate_once(self, symptom_text: str, symptom_list: List[str], prior_feedback: Optional[str], prior_result: Optional[Any]) -> Tuple[str, Optional[Any]]:
        symptom_list_json = json.dumps(symptom_list, ensure_ascii=False)
        
        # ---- Build the "previous result" block (if available)
        if isinstance(prior_result, (dict, list)):
            prior_result_str = json.dumps(prior_result, ensure_ascii=False, indent=2)
        elif isinstance(prior_result, str) and prior_result.strip():
            # Best-effort: if it's raw text, include it as-is
            prior_result_str = prior_result
        else:
            prior_result_str = ""

        prev_result_block = "" if not prior_result_str else f"""Previous attempt result (JSON):{prior_result_str}"""


        feedback_block = "" if not prior_feedback else f"\nPrevious attempt feedback:\n{prior_feedback}\n"
        
        user_prompt = gen_prompt(
            symptom_text=symptom_text,
            symptom_list=symptom_list_json,
            feedback_block=feedback_block,
            prev_result_block=prev_result_block
        )
        messages = [
            {"role": "system", "content": "You are a medical temporal sequence analysis assistant. Respond ONLY with a single JSON object in the specified shape."},
            {"role": "user", "content": user_prompt},
        ]

        raw, _resp = call_llm(
            messages,
            model=self.model_generate,
            text={"format": {"type": "json_object"}},   # <-- FIX: valid type
            temperature=0,
            print_debug=False
        )

        parsed = parse_json_best_effort(raw)
        # print("---- PARSED GENERATION ----")
        # print(parsed if parsed is not None else "(parse failed)")
        # print(raw)
        # print("--------------------------------\n")
        return raw, parsed

    # --- REPLACE: verify_once so it ALWAYS formats first and returns fixed ----------------
    def verify_once(self, symptom_text: str, symptom_list: List[str], current_result_raw_or_parsed: Any):
        """
        Always run format_check_and_fix first; then LLM verifies grouping/order/etc.
        Returns: (raw_llm_verify, parsed_llm_verify, fixed_list, fix_report)
        """
        # Normalize the list here to avoid character-soup bugs
        symptom_list = normalize_symptom_list(symptom_list)

        # 1) deterministic format/fix
        post = format_check_and_fix(current_result_raw_or_parsed, symptom_list)
        fixed_list = post["fixed"]
        fix_report = post["report"]

        # print("---- FORMAT CHECK & FIX REPORT ----")
        # print(json.dumps(fix_report, indent=2))
        # print("---- FIXED JSON (canonical) ----")
        # print(json.dumps({"temporal_result": fixed_list}, ensure_ascii=False, indent=2))
        # print("-----------------------------------\n")

        # 2) LLM verification on the FIXED JSON
        symptom_list_json = json.dumps(symptom_list, ensure_ascii=False)
        current_result_str = json.dumps({"temporal_result": fixed_list}, ensure_ascii=False)

        user_prompt = ver_prompt(
            symptom_text=symptom_text,
            symptom_list=symptom_list_json,
            initial_result=current_result_str
        )
        messages = [
            {"role": "system", "content": "You are a medical temporal sequence verification assistant. Respond ONLY with a single JSON object: {'score':<int>, 'feedback':<str>} (feedback required if score < 3)."},
            {"role": "user", "content": user_prompt},
        ]

        raw, _resp = call_llm(
            messages,
            model=self.model_verify,
            text={"format": {"type": "json_object"}},
            temperature=0,
            print_debug=False
        )
        parsed = parse_json_best_effort(raw)
        # print("---- PARSED VERIFICATION ----")
        # print(parsed if parsed is not None else "(parse failed)")
        # print("--------------------------------\n")
        return raw, parsed, fixed_list, fix_report



        # --- REPLACE: process_sample loop to normalize list and use FIXED JSON going forward --
    def process_sample(self, row, iterative: bool = True) -> Dict[str, Any]:
        """
        Verifier runs format_check_and_fix every round;
        the next generation uses the FIXED result (not the raw draft).
        """
        # get vaers id        
        vaers_id = row["vaers_id"]

        # Normalize the symptom list once here
        symptom_list = normalize_symptom_list(row["symptom_list"])
        symptom_text = row["symptom_text"]

        # ---------First: Temporal identification gate ---------
        temporal_info = None
        if self.use_temporal_gate:
            temporal_info, src = temporal_id_once(
                note_text=symptom_text,
                model=self.model_generate,  # reuse same model for simplicity
                min_confidence=self.temporal_conf_threshold,
                print_debug=True
            )
            # If no progression, skip extraction and return early
            if not temporal_info["has_progression"]:
                print("No temporal progression detected (gate). Skipping extraction.\n")
                return {
                    "temporal_id": temporal_info,
                    "attempts": [["No Temporal Information Found"]],
                    "final_result": None,   # not running extraction
                    "score": None,
                    "status": "no_temporal_pattern"
                }
        # -------------Then: Temporal Extraction------------------
        
        attempts = []
        rounds = self.max_iterations if iterative else 1
        feedback: Optional[str] = None
        prev_result_for_next_round: Optional[Any] = None

        for i in range(rounds):
            # print(f"\n########## ITERATION {i+1}/{rounds} ##########")
            # print(f"symptom_list type={type(symptom_list).__name__}, n={len(symptom_list)}; sample={symptom_list[:5]}")

            # 1) GENERATE (you already pass prior feedback/result)
            gen_raw, gen_parsed = self.generate_once(
                symptom_text, symptom_list,
                prior_feedback=feedback,
                prior_result=prev_result_for_next_round
            )
            candidate_for_verify = gen_parsed if gen_parsed is not None else gen_raw

            # 2) VERIFY (format-first inside)
            ver_raw, ver_parsed, fixed_list, fix_report = self.verify_once(
                symptom_text, symptom_list, candidate_for_verify
            )

            attempts.append({
                "raw_gen": gen_raw,
                "parsed_gen": gen_parsed,
                "raw_verify": ver_raw,
                "parsed_verify": ver_parsed,
                "post_fix": fixed_list,
                "fix_report": fix_report
            })

            score = ver_parsed.get("score") if isinstance(ver_parsed, dict) else None
            if isinstance(score, int) and score >= 3:
                # print(f"Verifier score >= 3 ({score}). Returning FIXED result.\n")
                return {
                    "attempts": attempts,
                    "final_result": {"temporal_result": fixed_list},
                    "score": score,
                    "status": "ok"
                }

            # 3) Not good enough — iterate with feedback; seed next round with FIXED result
            feedback = ver_parsed.get("feedback") if isinstance(ver_parsed, dict) else None
            prev_result_for_next_round = {"temporal_result": fixed_list}

        # 4) Max iterations — still return latest FIXED result
        return {
            "attempts": attempts,
            "final_result": {"temporal_result": fixed_list},
            "score": (ver_parsed.get("score") if isinstance(ver_parsed, dict) else 0),
            "status": "max_iterations_reached"
        }



In [ ]:
import os
# --------- Example batch driver ---------
agent = TemporalExtractionAgent(model_generate="claude-sonnet-4-5", max_iterations=4, use_temporal_gate=False)
all_rows = []
batch_size = 500
def _wrap(i, row):
    return i, agent.process_sample(row, iterative=True)

for i in range(0, len(sample), batch_size):
    # print
    print(f"Processing batch {i} to {i+batch_size}...")
    batch = sample.iloc[i:i+batch_size]
    rows = [row for _, row in batch.iterrows()]
    out = [None] * len(rows)
    # out = [None] * len(batch)
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(_wrap, i, row) for i, row in enumerate(rows)]
        for fut in concurrent.futures.as_completed(futures):
            idx, val = fut.result()
            out[idx] = val
        # results = list(executor.map(agent.process_sample([row for _, row in batch.iterrows()],iterative=True)))
    all_rows.extend(out)
    sample.loc[batch.index, 'new_llm_result'] = all_rows[-len(batch):]  # assign results to the correct rows

    sample.to_csv(f"claude_premium_updated_results_with_temporal_all_part{i}.csv", index=False)  # Save checkpoint



results_df = sample
results_df.info()

In [ ]:
len(all_rows)

In [ ]:
results_df.head()


In [ ]:
res_lst = []
for row in results_df.iterrows():
    final_result = row[1]['new_llm_result']['attempts'][-1]['post_fix']

    res = json.dumps(final_result, ensure_ascii=False, indent=2)
    res_lst.append(res)
results_df['final_result_cleaned'] = res_lst
results_df.head()
results_df.to_csv('claude_premium_updated_results_with_temporal_final.csv', index=False)


In [ ]:
for each in sample.llm_result_formatted.to_list():
    print("-------------------------------")
    print(each)
    break

# Evaluation

## Exact Match

In [ ]:
from typing import Iterable, Callable, Set, Hashable, Tuple, Any
# # exact match
# def strict_exact_match(
#     gold: Iterable[Iterable[Any]],
#     pred: Iterable[Iterable[Any]],
#     normalize: Optional[Callable[[Any], Any]] = None,
# ) -> Tuple[bool, str]:
#     """
#     Strict Exact Match for bucketed sequences:
#       - Compares two sequences of groups/buckets (e.g., [[A,B],[C],[D,E]]).
#       - Bucket order MUST match exactly.
#       - Within-bucket order is ignored (treated as sets).
#       - Duplicate items within a bucket are ignored (set semantics).

#     Args:
#         gold: Iterable of buckets (each bucket is an iterable of items).
#         pred: Iterable of buckets (each bucket is an iterable of items).
#         normalize: Optional normalization function applied to each item
#                    before comparison (e.g., str.lower, str.strip).

#     Returns:
#         (matched, reason)
#         matched: True if strictly equal under the bucket-as-set rule; else False
#         reason:  "" on success; otherwise a brief diagnostic message
#     """
#     # Materialize as list of lists for safe reuse
#     gold_buckets: List[List[Any]] = [list(b) for b in gold]
#     pred_buckets: List[List[Any]] = [list(b) for b in pred]

#     # 1) Same number of buckets
#     if len(gold_buckets) != len(pred_buckets):
#         return (False, f"Different # of buckets: gold={len(gold_buckets)} vs pred={len(pred_buckets)}")

#     # Helper: normalize + set-ify a bucket (ignore within-bucket order, dedupe)
#     def bucket_set(bucket: Iterable[Any]) -> Set[Any]:
#         if normalize is None:
#             return set(bucket)
#         return set(normalize(x) for x in bucket)

#     # 2) Compare each bucket as a set, in order
#     for i, (g_b, p_b) in enumerate(zip(gold_buckets, pred_buckets)):
#         g_set = bucket_set(g_b)
#         p_set = bucket_set(p_b)
#         if g_set != p_set:
#             missing = g_set - p_set
#             extra   = p_set - g_set
#             parts = []
#             if missing:
#                 parts.append(f"missing={sorted(missing)}")
#             if extra:
#                 parts.append(f"extra={sorted(extra)}")
#             details = "; ".join(parts) if parts else "bucket mismatch"
#             return (False, f"Bucket {i} differs: {details}")

#     # 3) All buckets matched
#     return (True, "")

from collections import Counter

def strict_exact_match(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
) -> Tuple[bool, str]:
    """
    Strict Exact Match for bucketed sequences:
      - Compares two sequences of groups/buckets (e.g., [[A,B],[C],[D,E]]).
      - Bucket order MUST match exactly.
      - Within-bucket order is ignored.
      - Duplicates MUST match (multiset/bag semantics), i.e., counts per item must be equal.

    Args:
        gold: Iterable of buckets (each bucket is an iterable of items).
        pred: Iterable of buckets (each bucket is an iterable of items).
        normalize: Optional normalization function applied to each item
                   before comparison (e.g., str.lower, str.strip).

    Returns:
        (matched, reason)
        matched: True if strictly equal under bucket-as-multiset; else False
        reason:  "" on success; otherwise a brief diagnostic message
    """
    # Materialize as list of lists for safe reuse
    gold_buckets = [list(b) for b in gold]
    pred_buckets = [list(b) for b in pred]

    # 1) Same number of buckets
    if len(gold_buckets) != len(pred_buckets):
        return (False, f"Different # of buckets: gold={len(gold_buckets)} vs pred={len(pred_buckets)}")

    # Normalizer
    norm = (lambda x: normalize(x)) if normalize is not None else (lambda x: x)

    # 2) Compare each bucket as a multiset (counts must match), in order
    for i, (g_b, p_b) in enumerate(zip(gold_buckets, pred_buckets)):
        g_ctr = Counter(norm(x) for x in g_b)
        p_ctr = Counter(norm(x) for x in p_b)
        if g_ctr != p_ctr:
            # Report missing/extra counts explicitly
            missing = g_ctr - p_ctr   # items required but missing counts
            extra   = p_ctr - g_ctr   # items present in pred but not in gold (or too many)
            parts = []
            if missing:
                parts.append(f"missing={dict(sorted(missing.items()))}")
            if extra:
                parts.append(f"extra={dict(sorted(extra.items()))}")
            details = "; ".join(parts) if parts else "bucket mismatch"
            return (False, f"Bucket {i} differs: {details}")

    # 3) All buckets matched
    return (True, "")

In [ ]:
# ---- Example ----
gold = [["fever", "headache"], ["rash"], ["fatigue", "nausea"]]
pred = [["rash"],["headache", "fever"], ["nausea", "fatigue"]]
strict_exact_match(gold, pred, normalize=lambda s: s.strip().lower())

In [ ]:
# ---- Example 1: Normalized match succeeds ----
gold = [["Dose 1", "Dose 2"], ["Observation", "Observation"]]
pred = [["dose 2", " dose 1 "], [" observation ", "Observation"]]
strict_exact_match(gold, pred, normalize=lambda s: s.strip().lower())

In [ ]:
# ---- Example 2: Bucket count mismatch ----
gold = [["fever"], ["rash"]]
pred = [["fever"], ["rash"], ["cough"]]
strict_exact_match(gold, pred, normalize=None)

In [ ]:
# ---- Example 3: Duplicate counts must match ----
gold = [["fever", "fever"], ["rash"]]
pred = [["fever"], ["rash"]]
strict_exact_match(gold, pred, normalize=None)

In [ ]:
# ---- Example 4: Bucket contents differ ----
gold = [["fever"], ["rash", "cough"]]
pred = [["fever"], ["rash", "hives"]]
strict_exact_match(gold, pred, normalize=None)

In [ ]:
# ------------------ Test Cases ------------------
tests = [
    {
        "name": "1) Perfect match; within-bucket order ignored",
        "gold": [['fever','headache'], ['rash'], ['fatigue','nausea']],
        "pred": [['headache','fever'], ['rash'], ['nausea','fatigue']],
    },
    {
        "name": "2) Different number of buckets",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever','headache'], ['rash'], ['fatigue']],
    },
    {
        "name": "3) Extra item in pred (bucket 0)",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever','headache','cough'], ['rash']],
    },
    {
        "name": "4) Missing item in pred (bucket 0)",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever'], ['rash']],
    },
    {
        "name": "5) Buckets swapped (order mismatch)",
        "gold": [['fever'], ['rash']],
        "pred": [['rash'], ['fever']],
    },
    {
        "name": "6) Duplicates within bucket are ignored",
        "gold": [['fever','fever','headache'], ['rash']],
        "pred": [['headache','fever'], ['rash']],
    },
    {
        "name": "7) Pred has an extra empty bucket",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever','headache'], ['rash'], []],
    },
    {
        "name": "8) Segmentation error across buckets",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever'], ['headache','rash']],
    },
    {
        "name": "9) Case/whitespace differences handled by normalize",
        "gold": [['Fever',' Headache '], ['RASH']],
        "pred": [['fever','headache'], ['rash']],
    },
]

# ------------------ Run Tests ------------------
for t in tests:
    match, reason = strict_exact_match(t["gold"], t["pred"], normalize=lambda s: s.strip().lower())
    print(f"{t['name']}\n  gold={t['gold']}\n  pred={t['pred']}\n  -> match={match}, reason={reason}\n")


[True, False, False, False, False, True, False, False, True]

## Group-aware Longest Common Contiguous Subsequence

In [ ]:

def group_aware_lccs(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
    return_alignment: bool = True,
) -> Tuple[int, float, float, float, Optional[Tuple[Tuple[int,int], Tuple[int,int]]]]:
    """
    Group-aware Longest Common Contiguous Subsequence (LCCS).

    Buckets are treated as sets (order-insensitive inside a bucket).
    We compute the longest block of consecutive buckets that appears in both sequences.

    Returns:
        (lccs_len, recall_gold, precision_pred, f1, alignment)
          - lccs_len: number of buckets in the longest contiguous match
          - recall_gold: lccs_len / max(1, len(gold_tokens))
          - precision_pred: lccs_len / max(1, len(pred_tokens))
          - f1: F1 over bucket-block matching using the above precision/recall
          - alignment: ((g_start, g_end_inclusive), (p_start, p_end_inclusive)) or None
    """
    def canon_bucket(bucket: Iterable[Any]) -> Hashable:
        if normalize is None:
            return frozenset(bucket)
        return frozenset(normalize(x) for x in bucket)

    gold_tokens: List[Hashable] = [canon_bucket(b) for b in gold]
    pred_tokens: List[Hashable] = [canon_bucket(b) for b in pred]
    m, n = len(gold_tokens), len(pred_tokens)

    # dp[i][j] = length of longest common suffix ending at gold[i-1], pred[j-1]
    dp = [[0]*(n+1) for _ in range(m+1)]
    best_len = 0
    best_end_g = 0
    best_end_p = 0

    for i in range(1, m+1):
        gi = gold_tokens[i-1]
        for j in range(1, n+1):
            if gi == pred_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
                if dp[i][j] > best_len:
                    best_len = dp[i][j]
                    best_end_g = i
                    best_end_p = j
            else:
                dp[i][j] = 0

    lccs_len = best_len
    recall_gold = lccs_len / max(1, m)
    precision_pred = lccs_len / max(1, n)
    f1 = (2 * recall_gold * precision_pred / (recall_gold + precision_pred)) if (recall_gold + precision_pred) else 0.0

    if not return_alignment or lccs_len == 0:
        return lccs_len, recall_gold, precision_pred, f1, None

    g_start = best_end_g - lccs_len
    g_end   = best_end_g - 1
    p_start = best_end_p - lccs_len
    p_end   = best_end_p - 1
    return lccs_len, recall_gold, precision_pred, f1, ((g_start, g_end), (p_start, p_end))




In [ ]:
# ---- Example ----
gold = [["fever","headache"], ["rash"], ["fatigue","nausea"]]
pred = [["headache","fever"], ["cough"], ["rash"], ["nausea","fatigue"]]
l, rg, pp, f1, span = group_aware_lccs(gold, pred, normalize=lambda s: s.strip().lower(), return_alignment=True)
print(l, rg, pp, f1, span)  # -> 2, 2/3, 2/4, ~0.57, ((1,2),(2,3))


In [ ]:
tests = [
    # 1
    {
        "name": "1) Perfect contiguous match",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['B','A'], ['C'], ['E','D']],
        "expected_len": 3
    },
    # 2
    {
        "name": "2) Middle bucket mismatch",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['B','A'], ['X'], ['E','D']],
        "expected_len": 1
    },
    # 3
    {
        "name": "3) Extra bucket inserted but 2-run remains",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['B','A'], ['C'], ['Y'], ['E','D']],
        "expected_len": 2
    },
    # 4
    {
        "name": "4) Buckets reordered",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['C'], ['B','A'], ['D','E']],
        "expected_len": 1
    },
    # 5
    {
        "name": "5) No common buckets",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['X'], ['Y'], ['Z']],
        "expected_len": 0
    },
    # 6
    {
        "name": "6) Two-bucket run at the end",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['X'], ['B','A'], ['C'], ['Z']],
        "expected_len": 2
    },
    # 7
    {
        "name": "7) Perfect run surrounded by extras",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['Y'], ['B','A'], ['C'], ['D','E'], ['Z']],
        "expected_len": 3
    },
    # 8
    {
        "name": "8) Split a gold bucket into two predicted buckets",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['B','A'], ['C'], ['D'], ['E']],
        "expected_len": 2
    },
    # 9
    {
        "name": "9) Merge adjacent gold buckets in prediction",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['A','B','C'], ['D','E']],
        "expected_len": 1
    },
    # 10
    {
        "name": "10) Case/whitespace differences handled by normalize",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [[' b ',' a '], [' c '], [' d ',' e ']],
        "expected_len": 3
    },
]

# ------------------ Run Tests ------------------
computed_lengths = []
for t in tests:
    l, rg, pp, f1, span = group_aware_lccs(
        t["gold"], t["pred"], normalize=lambda s: str(s).strip().lower(), return_alignment=True
    )
    computed_lengths.append(l)
    status = "PASS" if l == t["expected_len"] else "FAIL"
    print(f"{t['name']} -> LCCS length = {l} (expected {t['expected_len']}) [{status}]")

print("\nComputed LCCS lengths list:")
print(computed_lengths)

## Position Error Distance-aware

In [ ]:
from collections import defaultdict

def position_error_distance(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
    missing_penalty: str = "max",   # "ignore" | "max"
    return_details: bool = False,
) -> Tuple[float, float, Dict[str, Any]]:
    """
    Distance-aware Position Error for bucketed sequences (tie-aware).
    - Assign each item a bucket rank = its bucket index (0-based).
    - Compute absolute bucket-distance for items (ties inside a bucket share rank).
    - Normalize by the maximum possible distance (max(#buckets)-1) to map to [0,1].
    - Optionally penalize missing/extra items with worst-case distance.

    Args:
        gold: sequence of gold buckets, e.g., [["fever","headache"], ["rash"], ["fatigue","nausea"]]
        pred: sequence of predicted buckets
        normalize: optional function to normalize items before comparison (e.g., str.lower)
        missing_penalty: 
            - "ignore": compute over intersection only (no penalty for missing/extra)
            - "max": include union; items missing in one side get worst-case distance
        return_details: if True, returns a details dict with per-item distances and counts

    Returns:
        (score, mae_norm, details)
            score: 1 - mae_norm, clipped to [0,1]  (higher is better)
            mae_norm: normalized mean absolute error in [0,1]
            details: dict with diagnostics if return_details=True, else {}

    Notes:
        - Within-bucket order is ignored (tie-aware).
        - If an item appears in multiple buckets (shouldn't), earliest bucket index is used.
        - If both sequences have only 1 bucket, maximum possible distance is treated as 1 to avoid /0.
    """
    # Helper to normalize an item
    def norm(x: Any) -> Any:
        return x if normalize is None else normalize(x)

    # Map item -> earliest bucket index
    def item_to_rank(seq: Iterable[Iterable[Any]]) -> Dict[Any, int]:
        rank_map: Dict[Any, int] = {}
        for b_idx, bucket in enumerate(seq):
            for x in bucket:
                nx = norm(x)
                # keep earliest occurrence if duplicates across buckets
                if nx not in rank_map:
                    rank_map[nx] = b_idx
        return rank_map

    gold_ranks = item_to_rank(gold)
    pred_ranks = item_to_rank(pred)

    # Determine maximum possible distance between any two bucket ranks
    # B = max(len(list(gold)), len(list(pred)))  # we’ll reuse original iterables shortly; cast once more safely below
    
    # Re-materialize to avoid exhausting iterables
    gold_buckets: List[List[Any]] = [list(b) for b in gold]
    pred_buckets: List[List[Any]] = [list(b) for b in pred]
    B = max(len(gold_buckets), len(pred_buckets))
    max_dist = max(1, B - 1)  # avoid division by zero; if B==1, treat max_dist as 1

    gold_items = set(gold_ranks.keys())
    pred_items = set(pred_ranks.keys())
    inter = gold_items & pred_items
    only_gold = gold_items - pred_items
    only_pred = pred_items - gold_items

    distances: Dict[Any, int] = {}

    # Intersection: real distances
    for x in inter:
        d = abs(pred_ranks[x] - gold_ranks[x])
        distances[x] = d

    # Missing / extra handling
    if missing_penalty == "max":
        # assign worst-case distance to items missing on one side
        for x in only_gold:
            distances[x] = max_dist
        for x in only_pred:
            distances[x] = max_dist
    elif missing_penalty == "ignore":
        pass
    else:
        raise ValueError('missing_penalty must be "ignore" or "max"')

    if len(distances) == 0:
        has_any_gold = any(len(b) > 0 for b in gold_buckets)
        has_any_pred = any(len(b) > 0 for b in pred_buckets)

        # Case A: both sequences empty -> trivially perfect (no displacement possible)
        if not has_any_gold and not has_any_pred:
            score, mae_norm = 1.0, 0.0
            note = "both gold and pred empty; no displacement possible"
        else:
            # Case B: at least one side non-empty but no intersection -> worst score
            score, mae_norm = 0.0, 1.0
            note = "no shared items; intersection-based position treated as worst"

        details = {} if not return_details else {
            "num_gold_buckets": len(gold_buckets),
            "num_pred_buckets": len(pred_buckets),
            "counts": {
                "intersection": 0, 
                "only_gold": len(only_gold), 
                "only_pred": len(only_pred)
                },
            "max_dist": max_dist,
            "mae_raw": score,
            "per_item_distance": {},
            "note": note
        }
        return score, mae_norm, details

    mae = sum(distances.values()) / len(distances)
    mae_norm = min(1.0, mae / max_dist)
    score = max(0.0, 1.0 - mae_norm)

    details = {}
    if return_details:
        details = {
            "num_gold_buckets": len(gold_buckets),
            "num_pred_buckets": len(pred_buckets),
            "counts": {
                "intersection": len(inter),
                "only_gold": len(only_gold),
                "only_pred": len(only_pred),
                "evaluated_items": len(distances)
            },
            "max_dist": max_dist,
            "mae_raw": mae,
            "per_item_distance": dict(sorted(distances.items(), key=lambda kv: (-kv[1], str(kv[0])))),
            "note": None
        }

    return score, mae_norm, details


In [ ]:

# ----------------- Example -----------------
gold = [["fever","headache"], ["rash"], ["fatigue","nausea"]]
pred = [["headache","fever"], ["cough"], ["nausea","fatigue"]]
score, mae_norm, info = position_error_distance(
    gold, pred, normalize=lambda s: s.strip().lower(), missing_penalty="max", return_details=True
)
print(score, mae_norm)
print(info)

In [ ]:
def run_case(name, gold, pred):
    print(f"\n{name}")
    print("gold:", gold)
    print("pred:", pred)

    s_int, mae_int, d_int = position_error_distance(
        gold, pred, normalize=lambda s: str(s).strip().lower(), missing_penalty="ignore", return_details=True
    )
    print(f"  [intersection] score={s_int:.3f}, mae_norm={mae_int:.3f}, max_dist={d_int['max_dist']} | counts={d_int['counts']} | note = {d_int['note']}")
    print(f"    per-item distances (intersection): {d_int['per_item_distance']}")

    s_uni, mae_uni, d_uni = position_error_distance(
        gold, pred, normalize=lambda s: str(s).strip().lower(), missing_penalty="max", return_details=True
    )
    print(f"  [union]        score={s_uni:.3f}, mae_norm={mae_uni:.3f}, max_dist={d_uni['max_dist']} | counts={d_uni['counts']}| note = {d_int['note']}")
    print(f"    per-item distances (union): {d_uni['per_item_distance']}")


# ------------------ Tests ------------------
gold1 = [["fever","headache"], ["rash"], ["fatigue","nausea"]]
pred1 = [["headache","fever"], ["rash"], ["nausea","fatigue"]]
run_case("Test 1: perfect match", gold1, pred1)

pred2 = [["headache"], ["fever","rash"], ["nausea","fatigue"]]
run_case("Test 2: one item shifted by 1 bucket", gold1, pred2)

pred3 = [["headache"], ["rash"], ["nausea","fatigue","fever"]]
run_case("Test 3: one item shifted by 2 buckets", gold1, pred3)

pred4 = [["headache","fever"], [], ["nausea","fatigue","cough"]]
run_case("Test 4: missing gold item + extra pred item", gold1, pred4)

pred5 = [["headache","fever"], ["rash","cough"], ["nausea","fatigue"]]
run_case("Test 5: extra item only", gold1, pred5)

gold6 = [["a","b"], ["c"], ["d","e"]]
pred6 = [["a","b"], ["d","e"], ["c"]]
run_case("Test 6: swap two buckets", gold6, pred6)

gold7 = [["a","b"], ["c"]]
pred7 = [["a","c"], ["b","c"]]
run_case("Test 7: duplicate in pred (earliest occurrence used)", gold7, pred7)

gold8 = [[" Fever ","Headache "], ["RASH"]]
pred8 = [["fever","headache"], ["rash"]]
run_case("Test 8: normalization check", gold8, pred8)

gold9 = [["a","b","c"]]
pred9 = [["1","2","3","x"]]
run_case("Test 9: single-bucket edge case", gold9, pred9)

gold10 = [["a"], ["b"], ["c"], ["d"]]
pred10 = [["a","b","c","d"]]
run_case("Test 10: different #buckets changes normalization", gold10, pred10)

gold11 = []
pred11 = []
run_case("Test 11: empty for both", gold11, pred11)

# 12: non-empty but no intersection
gold12 = [["a"], ["b"]]
pred12 = [["x"], ["y"]]
run_case("Test 12: non-empty but no intersection", gold12, pred12)

# 13: one empty, one non-empty
gold13 = []
pred13 = [["a"], ["b"]]
run_case("Test 13: one empty, one non-empty", gold13, pred13)



## Kendall-Tau-b 

In [ ]:
import math
from itertools import combinations

def _norm_fn(normalize: Optional[Callable[[Any], Any]]):
    return (lambda x: x) if normalize is None else normalize

def _item_set(seq: Iterable[Iterable[Any]], normalize: Optional[Callable[[Any], Any]]) -> set:
    """All unique items across buckets (order ignored)."""
    norm = _norm_fn(normalize)
    return {norm(x) for bucket in seq for x in bucket}

def _item_ranks(seq: Iterable[Iterable[Any]], normalize: Optional[Callable[[Any], Any]]) -> Dict[Any, int]:
    """
    Map item -> bucket index (rank). Within-bucket order is ignored.
    Assumes no cross-bucket duplicates; if they exist, earliest index wins.
    """
    norm = _norm_fn(normalize)
    ranks: Dict[Any, int] = {}
    for b_idx, bucket in enumerate(seq):
        seen = set()
        for x in bucket:
            k = norm(x)
            if k in seen:   # ignore within-bucket duplicates
                continue
            seen.add(k)
            if k not in ranks:
                ranks[k] = b_idx
    return ranks

# ---------------- Coverage: item set P/R/F1 ----------------

def item_set_prf1(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
) -> Tuple[float, float, float, Dict[str, int]]:
    """Coverage only (ignores order)."""
    gold_items = _item_set(gold, normalize)
    pred_items = _item_set(pred, normalize)
    inter = gold_items & pred_items

    precision = len(inter) / len(pred_items) if pred_items else 1.0
    recall    = len(inter) / len(gold_items) if gold_items else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    details = {
        "gold_count": len(gold_items),
        "pred_count": len(pred_items),
        "intersection": len(inter),
        "only_gold": len(gold_items - pred_items),
        "only_pred": len(pred_items - gold_items),
    }
    return precision, recall, f1, details

# ---------------- Order: tie-aware Kendall’s τ-b (intersection) ----------------

def kendall_tau_b_ties(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
    return_details: bool = False,
) -> Tuple[float, Dict[str, Any]]:
    """
    Tie-aware Kendall's τ-b on bucketed sequences.
    - Rank = bucket index; items in same bucket are 'tied'.
    - Only items present in BOTH are compared (pure ordering on intersection).
    """
    gold_r = _item_ranks(gold, normalize)
    pred_r = _item_ranks(pred, normalize)

    items = sorted(gold_r.keys() & pred_r.keys(), key=lambda x: str(x))
    n = len(items)
    if n < 2:
        return 1.0, ({"num_items_intersection": n, "pairs": 0} if return_details else {})

    C = D = T_g = T_p = 0  # concordant, discordant, ties in gold, ties in pred
    disagreements: List[Tuple[Any, Any, int, int]] = []

    for i in range(n):
        a = items[i]
        ga = gold_r[a]; pa = pred_r[a]
        for j in range(i+1, n):
            b = items[j]
            gb = gold_r[b]; pb = pred_r[b]

            # relations: -1 (a before b), 0 (tie), +1 (a after b)
            rg = -1 if ga < gb else (1 if ga > gb else 0)
            rp = -1 if pa < pb else (1 if pa > pb else 0)

            if rg == 0: T_g += 1
            if rp == 0: T_p += 1

            if rg != 0 and rp != 0:
                if rg == rp:
                    C += 1
                else:
                    D += 1
                    if len(disagreements) < 20:
                        disagreements.append((a, b, rg, rp))

    numer = C - D
    denom_left  = C + D + T_g
    denom_right = C + D + T_p
    denom = math.sqrt(denom_left * denom_right) if denom_left > 0 and denom_right > 0 else 0.0
    tau_b = (numer / denom) if denom > 0 else 1.0

    if not return_details:
        return tau_b, {}

    return tau_b, {
        "num_items_intersection": n,
        "pairs": n * (n - 1) // 2,
        "C": C, "D": D, "T_g": T_g, "T_p": T_p,
        "numerator": numer,
        "denominator_parts": {"left": denom_left, "right": denom_right},
        "tau_b": tau_b,
        "sample_disagreements": disagreements,
        "relation_legend": {"-1": "a precedes b", "0": "tie", "+1": "a follows b"},
    }

# ---------------- Optional: combine Order × Coverage ----------------

def combined_order_coverage(
    tau_b: float,
    item_f1: float,
    method: str = "product",   # "product" | "geom" | "weighted_geom"
    weight_tau: float = 0.6,   # only used for "weighted_geom"
    weight_f1: float = 0.4,
) -> float:
    """
    Roll up order (τ-b) and coverage (item F1) into one number in [0,1].
    """
    tau = max(0.0, min(1.0, tau_b))
    f1  = max(0.0, min(1.0, item_f1))
    if method == "product":
        return tau * f1
    if method == "geom":
        return math.sqrt(tau * f1)
    if method == "weighted_geom":
        wsum = max(1e-9, weight_tau + weight_f1)
        # (tau^w1 * f1^w2)^(1/(w1+w2))
        return (tau ** weight_tau) * (f1 ** weight_f1) ** (1.0 / wsum)
    raise ValueError('method must be "product", "geom", or "weighted_geom"')

# ---------------- Convenience wrapper ----------------

def evaluate_order_and_coverage(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
    combine: Optional[str] = None,  # None | "product" | "geom" | "weighted_geom"
) -> Dict[str, Any]:
    """
    Returns:
      - tau_b (order on intersection, tie-aware)
      - item_precision, item_recall, item_f1 (coverage)
      - combined (optional)
      - details (diagnostics)
    """
    tau, tau_info = kendall_tau_b_ties(gold, pred, normalize=normalize, return_details=True)
    P, R, F1, cov_info = item_set_prf1(gold, pred, normalize=normalize)

    out = {
        "tau_b": tau,
        "item_precision": P,
        "item_recall": R,
        "item_f1": F1,
        "order_details": tau_info,
        "coverage_details": cov_info,
    }
    if combine:
        out["combined"] = combined_order_coverage(tau, F1, method=combine)
        out["combined_method"] = combine
    return out




In [ ]:
# ---------------- Example ----------------
gold = [["Fever","Headache"], ["Rash"], ["Fatigue","Nausea"]]
pred = [["headache","fever"], ["Cough"], ["nausea","fatigue"]]
res = evaluate_order_and_coverage(
    gold, pred, normalize=lambda s: s.strip().lower(), combine="product"
)
print(res)

## Sanity Check

In [ ]:
# # =========================
# # LCCS, Kendall's tau-b, and PED unit checks (3-bucket cases)
# # Assumes these functions already exist in the notebook scope:
# #   group_aware_lccs, evaluate_order_and_coverage, position_error_distance
# # =========================

# import pandas as pd
# import math

# def get_tau_b(oc: dict) -> float:
#     """Pull tau_b from your evaluate_order_and_coverage return dict."""
#     if isinstance(oc, dict) and "tau_b" in oc and isinstance(oc["tau_b"], (int, float)):
#         return float(oc["tau_b"])
#     od = oc.get("order_details", {}) if isinstance(oc, dict) else {}
#     if isinstance(od, dict) and "tau_b" in od and isinstance(od["tau_b"], (int, float)):
#         return float(od["tau_b"])
#     return float("nan")

# # ---------- Test set ----------
# # Notation:
# #   Gold groups: G1=[...], G2=[...], G3=[...]
# #   Pred  groups: P1=[...], P2=[...], P3=[...]
# # Items: A,B,C,D,E
# # For τ-b calculations:
# #   n = 5 items -> total unordered pairs n0 = 5*4/2 = 10
# #   S = C - D (concordant minus discordant over NON-tied pairs)
# #   n1 = ties in gold (sum over groups of comb(size,2))
# #   n2 = ties in pred
# #   τ_b = S / sqrt((n0 - n1) * (n0 - n2))
# #
# # For LCCS (group-aware but item-based contiguous subsequence intuition):
# #   We flatten to item sequences respecting group order & within-group order as written below,
# #   then take the Longest Common Contiguous Subsequence length = L.
# #   precision = L / len(pred_items), recall = L / len(gold_items), F1 = 2PR/(P+R).
# #
# # For PED (Position Error Distance; illustrative/ideal MAE over group indices):
# #   distance for an item = |group_index_pred - group_index_gold|
# #   MAE = sum(distances)/count(items)
# #   (Your function returns two floats; we show the ideal MAE as a reference.)

# tests = [

#     # 1) Perfect match
#     dict(
#         name="X1_perfect",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B"], ["C"], ["D","E"]],
#         note="""
# τ-b: perfect agreement => C=8, D=0, ties in both: n1=n2=2 (AB, DE), n0=10
#      τ_b = (8-0)/sqrt((10-2)*(10-2)) = 8/8 = 1.0
# LCCS: gold=[A,B,C,D,E], pred=[A,B,C,D,E] => L=5, P=1, R=1, F1=1
# PED:  all items stay in the same group => MAE=0
# """
#     ),

#     # 2) Swap the last two groups (adjacent swap)
#     dict(
#         name="X2_swap_G2_G3",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B"], ["D","E"], ["C"]],
#         note="""
# τ-b: pairs (A,B) & (D,E) are ties in both => n1=n2=2; n0=10
#      Check signs: only (C vs D) and (C vs E) flip => C=7, D=2 -> S=5
#      τ_b = 5 / sqrt((10-2)*(10-2)) = 5/8 = 0.625
# LCCS: gold seq [A,B,C,D,E], pred seq [A,B,D,E,C] => longest common contiguous block is [A,B] (L=2)
#      P=2/5=0.4, R=2/5=0.4, F1=0.4
# PED:  group indices: A,B:1→1 (0); C:2→3 (1); D,E:3→2 (1 each)
#      MAE = (0+0+1+1+1)/5 = 0.6
# """
#     ),

#     # 3) Merge C with D,E into one group in pred
#     dict(
#         name="X3_merge_tail",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B"], ["C","D","E"]],
#         note="""
# τ-b: n0=10; gold ties (AB, DE) => n1=2; pred ties (AB plus C,D,E all tied except AB) => n2=4
#      Non-tied both: A/B vs C/D/E (6 pairs) all concordant; others tied in one ranking
#      S=6 => τ_b = 6 / sqrt((10-2)*(10-4)) = 6 / sqrt(8*6) = 6/√48 ≈ 0.8660254
# LCCS: gold [A,B,C,D,E], pred [A,B,C,D,E] (contiguous order preserved) => L=5, P=R=F1=1
# PED:  D,E moved from group 3 -> 2 (1 each); others unchanged => MAE=(0+0+0+1+1)/5 = 0.4
# """
#     ),

#     # 4) Split first group in pred (A alone), tie C with D,E => bigger tie set in pred
#     dict(
#         name="X4_split_head",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A"], ["B"], ["C","D","E"]],
#         note="""
# τ-b: n0=10; gold ties: (AB, DE) => n1=2; pred ties: (C,D,E) => 3 pairs, n2=3
#      Concordant: A vs C/D/E (3), B vs C/D/E (3) => C=6, D=0 => S=6
#      τ_b = 6 / sqrt((10-2)*(10-3)) = 6 / √56 ≈ 0.8017837
# LCCS: gold [A,B,C,D,E], pred [A,B,C,D,E] => L=5, P=R=F1=1
# PED:  B: 1→2 (1); C: 2→3 (1); A,D,E unchanged (0) => MAE=(0+1+1+0+0)/5 = 0.4
# """
#     ),

#     # 5) Strong reversal of order (all groups reversed)
#     dict(
#         name="X5_full_reverse",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["D","E"], ["C"], ["A","B"]],
#         note="""
# τ-b: complete reversal except within-group ties; n0=10; n1=n2=2
#      All 8 cross-group pairs discordant => C=0, D=8 => S=-8
#      τ_b = -8 / sqrt((10-2)*(10-2)) = -8/8 = -1.0
# LCCS: gold [A,B,C,D,E], pred [D,E,C,A,B] => best contiguous match is [C] => L=1
#      P=R=1/5=0.2, F1=0.2
# PED:  A,B: 1→3 (2 each); C: 2→2 (0); D,E: 3→1 (2 each) => MAE=(2+2+0+2+2)/5 = 1.6
# """
#     ),

#     # 6) Move C up into the first group in pred (order preserved in items)
#     dict(
#         name="X6_pull_C_up",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B","C"], ["D"], ["E"]],
#         note="""
# τ-b: n0=10; gold ties: (AB, DE) => n1=2; pred ties: (A,B,C) => 3 pairs, n2=3
#      Non-tied both: A/B vs D/E (4) + C vs D/E (2) => C=6, D=0 => S=6
#      τ_b = 6 / sqrt(8*7) ≈ 0.8017837
# LCCS: gold [A,B,C,D,E], pred [A,B,C,D,E] => L=5, P=R=F1=1
# PED:  C: 2→1 (1); D,E: 3→2 (1 each); A,B unchanged
#      MAE = (0+0+1+1+1)/5 = 0.6
# """
#     ),

#     # 7) Slight mis-group: move E up one bucket (keeps order otherwise)
#     dict(
#         name="X7_move_E_up_one",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B"], ["C","E"], ["D"]],
#         note="""
# τ-b: n0=10; gold ties: (AB, DE) => n1=2; pred ties: (AB) only => n2=1
#      Concordant: A/B vs all (6) + C vs D (1) => C=7, D=0 => S=7
#      τ_b = 7 / sqrt((10-2)*(10-1)) = 7 / √72 ≈ 0.825
# LCCS: gold [A,B,C,D,E], pred [A,B,C,E,D] => best contiguous match [A,B,C] => L=3
#      P=R=3/5=0.6, F1=0.6
# PED:  E: 3→2 (1); others unchanged => MAE=(0+0+0+0+1)/5 = 0.2
# """
#     ),

#     # 8) Cross misplacements: B pushed to last bucket; D up to middle (both sides change)
#     dict(
#         name="X8_cross_moves",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A"], ["C","D"], ["B","E"]],
#         note="""
# τ-b: n0=10; gold ties: (AB, DE) => n1=2; pred ties: (CD) and (BE) => n2=2
#      Pair-wise:
#        - A vs C/D/E: A precedes all in both => 3 concordant
#        - B vs C/D: B precedes (gold), but follows in pred (group3 vs group2) => 2 discordant
#        - B vs E: tie in pred & gold? (gold: B vs E is B precedes E; pred: tie (BE)) => tie in pred only
#        - C vs E: C precedes E in both => 1 concordant
#        - A vs B: tie gold only; C vs D: tie pred only; D vs E: tie gold only
#      Totals: C=4, D=2 => S=2
#      τ_b = 2 / sqrt((10-2)*(10-2)) = 2/8 = 0.25
# LCCS: gold [A,B,C,D,E], pred [A,C,D,B,E] => best contiguous match [C,D] => L=2
#      P=R=2/5=0.4, F1=0.4
# PED:  A:1→1 (0); B:1→3 (2); C:2→2 (0); D:3→2 (1); E:3→3 (0)
#      MAE = (0+2+0+1+0)/5 = 0.6
# """
#     ),
# ]

# # ---------- Runner ----------
# rows = []
# for t in tests:
#     gold = t["gold"]
#     pred = t["pred"]

#     # LCCS
#     L, P, R, F1, _ = group_aware_lccs(gold, pred, normalize=None, return_alignment=False)
#     # τ-b (extract from ord+cov)
#     oc = evaluate_order_and_coverage(gold, pred, normalize=None, combine=None)
#     tau = get_tau_b(oc)
#     # PED
#     v0, v1, _ = position_error_distance(gold, pred, normalize=None, missing_penalty="max", return_details=False)

#     rows.append({
#         "test": t["name"],
#         "lccs_L": L, "lccs_P": P, "lccs_R": R, "lccs_F1": F1,
#         "tau_b": tau,
#         "ped_v0": v0, "ped_v1": v1,
#         "gold": gold, "pred": pred,
#         "ideal_calc": t["note"].strip()
#     })

# df = pd.DataFrame(rows, columns=[
#     "test","lccs_L","lccs_P","lccs_R","lccs_F1","tau_b","ped_v0","ped_v1","gold","pred","ideal_calc"
# ])
# df


## evaluation pipeline

In [ ]:
# ---------------- Normalization ----------------
def _normalize_default(x: Any) -> str:
    return str(x).strip().lower()

# ---------------- Coercion to buckets ----------------

def _coerce_to_buckets(raw: Any, normalize: Optional[Callable[[Any], Any]] = None) -> List[List[Any]]:
    """
    Convert model outputs into List[List[item]] buckets.
    Handles:
      - list[list[str]]
      - list[str] (single bucket)
      - list[dict] (each dict is a bucket: keys are symptoms; ignore values that are "none"/empty)
      - dict with keys among {'groups','buckets','sequence','timeline','temporal_groups',
                              'ordered_groups','result','temporal_result','output'}
      - JSON string or Python-literal string of any of the above
      - Plain text 'A,B | C | D,E'
    """
    norm = (lambda x: x) if normalize is None else normalize

    def _dedupe_bucket(b: Iterable[Any]) -> List[Any]:
        seen, out = set(), []
        for x in b:
            k = norm(x)
            if k in seen or k == "":
                continue
            seen.add(k)
            out.append(k)
        return out

    def _is_none_val(v: Any) -> bool:
        if v is None: return True
        if isinstance(v, str) and norm(v) == "none": return True
        if isinstance(v, (list, tuple, set, dict)) and len(v) == 0: return True
        return False

    # ---------- STRING INPUT: try JSON, then Python literal, then fallback ----------
    if isinstance(raw, str):
        s = raw.strip()

        # 1) JSON attempt
        if s.startswith("{") or s.startswith("[") or s.startswith("```"):
            try:
                obj = json.loads(s.strip("`"))
                return _coerce_to_buckets(obj, normalize=normalize)
            except Exception:
                # 2) Python literal (handles single quotes)
                try:
                    obj = ast.literal_eval(s)
                    return _coerce_to_buckets(obj, normalize=normalize)
                except Exception:
                    pass

        # 3) Fallback: 'A,B | C | D,E' style
        groups = re.split(r"\|\||\||\n+", s)
        buckets = []
        for g in groups:
            toks = [t.strip() for t in re.split(r"[,\t;]+", g) if t.strip()]
            bk = _dedupe_bucket(toks)
            if bk:
                buckets.append(bk)
        return buckets

    # ---------- PYTHON OBJECTS ----------
    if isinstance(raw, (list, tuple)):
        raw = list(raw)
        if len(raw) == 0:
            return []

        # list of dicts -> each dict is a bucket; keep keys whose values are not "none"/empty
        if all(isinstance(g, dict) for g in raw):
            out = []
            for bucket_obj in raw:
                items = [k for k, v in bucket_obj.items() if not _is_none_val(v)]
                bk = _dedupe_bucket(items)
                if bk:
                    out.append(bk)
            return out

        # list of lists/sets
        if all(isinstance(g, (list, tuple, set)) for g in raw):
            return [_dedupe_bucket(list(g)) for g in raw if len(_dedupe_bucket(list(g))) > 0]

        # list of scalars -> single bucket
        if all(not isinstance(g, (list, tuple, set, dict)) for g in raw):
            b = _dedupe_bucket(raw)
            return [b] if b else []

        # mixed: coerce element-wise
        out = []
        for g in raw:
            if isinstance(g, dict):
                items = [k for k, v in g.items() if not _is_none_val(v)]
                bk = _dedupe_bucket(items)
            elif isinstance(g, (list, tuple, set)):
                bk = _dedupe_bucket(list(g))
            else:
                bk = _dedupe_bucket([g])
            if bk:
                out.append(bk)
        return out

    if isinstance(raw, dict):
        # envelopes
        for key in ["groups","buckets","sequence","timeline","temporal_groups",
                    "ordered_groups","result","temporal_result","output"]:
            if key in raw:
                return _coerce_to_buckets(raw[key], normalize=normalize)
        # dict of symptom->mentions (single bucket)
        items = [k for k, v in raw.items() if not _is_none_val(v)]
        b = _dedupe_bucket(items)
        return [b] if b else []

    # Last resort
    return []


def evaluate_temporal_sample(
    gold_raw: Any,
    pred_raw: Any,
    normalize: Optional[Callable[[Any], Any]] = _normalize_default,
    compute_union_position: bool = True,  # also compute order+coverage position score
    lccs_alignment: bool = True,
    combine_method: Optional[str] = "product",  # None | "product" | "geom" | "weighted_geom"
) -> Dict[str, Any]:
    """
    Parse gold/pred, then compute:
      - Exact Match
      - Item coverage P/R/F1
      - Kendall tau-b (ties) on intersection
      - Position Error (intersection) and (optional) union
      - LCCS (len, Rec_G, Prec_P, F1, alignment)
      - Combined = tau_b x item_F1 (if combine_method is not None)
    """
    gold = _coerce_to_buckets(gold_raw, normalize=normalize)
    pred = _coerce_to_buckets(pred_raw, normalize=normalize)

    # print("Coerced Gold:", gold)
    # print("Coerced Pred:", pred)
    # Exact match
    em_ok, em_reason = strict_exact_match(gold, pred, normalize=normalize)

    # Coverage (items only)
    item_P, item_R, item_F1, cov_details = item_set_prf1(gold, pred, normalize=normalize)

    # Kendall tau-b (ties)
    tau_b, tau_details = kendall_tau_b_ties(gold, pred, normalize=normalize, return_details=True)

    # Position error (intersection-only)
    pos_score_int, pos_mae_norm_int, pos_details_int = position_error_distance(
        gold, pred, normalize=normalize, missing_penalty="ignore", return_details=True
    )
    # Position error (union / order+coverage)
    pos_score_union, pos_mae_norm_union, pos_details_union = None, None, None
    if compute_union_position:
        pos_score_union, pos_mae_norm_union, pos_details_union = position_error_distance(
            gold, pred, normalize=normalize, missing_penalty="max", return_details=True
        )

    # LCCS (contiguous)
    lccs_len, lccs_rec_g, lccs_prec_p, lccs_f1, lccs_align = group_aware_lccs(
        gold, pred, normalize=normalize, return_alignment=lccs_alignment
    )

    # Combined (order x coverage)
    combined = None
    if combine_method is not None:
        # Simple product by default: tau_b * item_F1
        if combine_method == "product":
            combined = tau_b * item_F1
        elif combine_method == "geom":
            combined = (tau_b * item_F1) ** 0.5
        elif combine_method == "weighted_geom":
            # default weights tau:0.6, f1:0.4
            w_tau, w_f1 = 0.6, 0.4
            denom = max(1e-9, w_tau + w_f1)
            combined = ((tau_b ** w_tau) * (item_F1 ** w_f1)) ** (1.0 / denom)
        else:
            raise ValueError("combine_method must be one of: None | 'product' | 'geom' | 'weighted_geom'")

    return {
        "exact_match": {"match": em_ok, "reason": em_reason},
        "coverage": {"precision": item_P, "recall": item_R, "f1": item_F1, "details": cov_details},
        "kendall_tau_b": {"tau_b": tau_b, "details": tau_details},
        "position_error_intersection": {
            "score": pos_score_int, "mae_norm": pos_mae_norm_int, "details": pos_details_int
        },
        "position_error_union": None if pos_score_union is None else {
            "score": pos_score_union, "mae_norm": pos_mae_norm_union, "details": pos_details_union
        },
        "lccs": {
            "length": lccs_len, "recall_gold": lccs_rec_g, "precision_pred": lccs_prec_p,
            "f1": lccs_f1, "alignment": lccs_align
        },
        "combined_order_x_coverage": {"method": combine_method, "score": combined},
        "canonical_inputs": {"gold": gold, "pred": pred},  # after coercion/normalization
    }


# pred_example = results_df.final_result_cleaned.to_list()[0]
# gold_example = sample.llm_result_formatted.to_list()[0]

# # print(pred_example)
# # print(gold_example)

# res = evaluate_temporal_sample(
#     gold_example, pred_example,
#     normalize=lambda s: str(s).strip().lower(),
#     compute_union_position=True,
#     lccs_alignment=True,
#     combine_method="product",  # tau_b * item_F1
# )
# # from pprint import pprint
# # pprint(res)

In [ ]:
from typing import Any, Callable, Dict, List, Optional, Tuple

# Assumes these exist from earlier messages:
# - evaluate_temporal_sample(gold_raw, pred_raw, normalize=..., compute_union_position=True, lccs_alignment=True, combine_method="product")

def mean_safe(vals: List[Optional[float]]) -> Optional[float]:
    nums = [v for v in vals if v is not None]
    return sum(nums)/len(nums) if nums else None

def evaluate_temporal_batch(
    gold_list: List[Any],
    pred_list: List[Any],
    normalize: Optional[Callable[[Any], Any]] = lambda s: str(s).strip().lower(),
    compute_union_position: bool = True,
    lccs_alignment: bool = False,     # turn on if you want alignments per sample
    combine_method: Optional[str] = "product",  # None | "product" | "geom" | "weighted_geom"
) -> Dict[str, Any]:
    """
    Evaluate multiple (gold, pred) pairs.
    Returns per-sample metrics and macro/micro averages across all successfully evaluated samples.
    """
    if len(gold_list) != len(pred_list):
        raise ValueError(f"gold_list and pred_list must have same length; got {len(gold_list)} vs {len(pred_list)}")

    per_sample: List[Dict[str, Any]] = []
    n = len(gold_list)

    # Micro-coverage accumulators
    micro_gold_total = 0
    micro_pred_total = 0
    micro_inter_total = 0

    # For macro means
    ems, tau_bs = [], []
    cov_Ps, cov_Rs, cov_F1s = [], [], []
    pos_int_scores, pos_int_maes = [], []
    pos_union_scores, pos_union_maes = [], []
    lccs_lens, lccs_recalls, lccs_precs, lccs_f1s = [], [],[], []
    combined_scores = []

    failures = 0
    # counter = 2

    for idx, (g_raw, p_raw) in enumerate(zip(gold_list, pred_list)):
        try:
            res = evaluate_temporal_sample(
                g_raw, p_raw,
                normalize=normalize,
                compute_union_position=compute_union_position,
                lccs_alignment=lccs_alignment,
                combine_method=combine_method,
            )
            per_sample.append({"index": idx, **res})

            # Exact match rate
            ems.append(1.0 if res["exact_match"]["match"] else 0.0)
            # print('ems: ',ems)
            # Coverage (macro)
            cov = res["coverage"]
            cov_Ps.append(cov["precision"])
            cov_Rs.append(cov["recall"])
            cov_F1s.append(cov["f1"])
            # print('coverage: ',cov_F1s)
            # Coverage (micro)
            cd = cov["details"]
            micro_gold_total += cd["gold_count"]
            micro_pred_total += cd["pred_count"]
            micro_inter_total += cd["intersection"]

            # Kendall tau-b
            tau_bs.append(res["kendall_tau_b"]["tau_b"])
            # print('tau_bs: ',tau_bs)
            # Position error (intersection)
            posi = res["position_error_intersection"]
            pos_int_scores.append(posi["score"])
            pos_int_maes.append(posi["mae_norm"])

            # Position error (union) if present
            if res["position_error_union"] is not None:
                posu = res["position_error_union"]
                pos_union_scores.append(posu["score"])
                pos_union_maes.append(posu["mae_norm"])
                # print('pos_union_scores: ',pos_union_scores)
            # LCCS
            lccs = res["lccs"]
            lccs_lens.append(lccs["length"])
            lccs_recalls.append(lccs["recall_gold"])
            lccs_precs.append(lccs["precision_pred"])
            lccs_f1s.append(lccs["f1"])
            # print('lccs_f1s: ',lccs_f1s)
            
            # Combined
            if res["combined_order_x_coverage"]["score"] is not None:
                combined_scores.append(res["combined_order_x_coverage"]["score"])
            
        except Exception as e:
            failures += 1
            per_sample.append({
                "index": idx,
                "error": str(e),
                "raw_gold": g_raw,
                "raw_pred": p_raw,
            })
            # skip from averages

    evaluated = n - failures

    # Macro averages
    macro = {
        "exact_match_rate": mean_safe(ems),
        "coverage_macro": {
            "precision": mean_safe(cov_Ps),
            "recall":    mean_safe(cov_Rs),
            "f1":        mean_safe(cov_F1s),
        },
        "kendall_tau_b": mean_safe(tau_bs),
        "position_error_intersection": {
            "score":   mean_safe(pos_int_scores),
            "mae_norm":mean_safe(pos_int_maes),
        },
        "position_error_union": None if not pos_union_scores else {
            "score":   mean_safe(pos_union_scores),
            "mae_norm":mean_safe(pos_union_maes),
        },
        "lccs": {
            "length":          mean_safe(lccs_lens),
            "recall_gold":     mean_safe(lccs_recalls),
            "precision_pred":  mean_safe(lccs_precs),
            "f1":              mean_safe(lccs_f1s),
        },
        "combined_order_x_coverage": mean_safe(combined_scores) if combined_scores else None,
    }

    # Micro coverage (sums → single P/R/F1)
    if micro_pred_total == 0 and micro_gold_total == 0:
        micro_cov = {"precision": None, "recall": None, "f1": None,
                     "totals": {"gold": 0, "pred": 0, "intersection": 0}}
    else:
        micro_P = (micro_inter_total / micro_pred_total) if micro_pred_total else 0.0
        micro_R = (micro_inter_total / micro_gold_total) if micro_gold_total else 0.0
        micro_F1 = (2 * micro_P * micro_R / (micro_P + micro_R)) if (micro_P + micro_R) else 0.0
        micro_cov = {
            "precision": micro_P,
            "recall": micro_R,
            "f1": micro_F1,
            "totals": {"gold": micro_gold_total, "pred": micro_pred_total, "intersection": micro_inter_total}
        }

    return {
        "n_samples": n,
        "n_evaluated": evaluated,
        "n_failed": failures,
        "macro_averages": macro,
        "micro_coverage": micro_cov,
        "per_sample": per_sample,
    }




In [ ]:
# Each entry can be list-of-lists, JSON, dicts with 'groups'/'buckets', or simple "A,B | C | D,E" text.
pred_example = results_df.final_result_cleaned.to_list()
gold_example = sample.llm_result_formatted.to_list()

summary = evaluate_temporal_batch(
    gold_example, pred_example,
    normalize=lambda s: str(s).strip().lower(),
    compute_union_position=True,
    lccs_alignment=False,
    combine_method="product",
)

from pprint import pprint
pprint(summary)